# CausalMan: RCA data generation

This notebook generates the interventional datasets for the CausalMan root-cause analysis (RCA) benchmark.
There are four tasks, each run on two scales, giving eight datasets in total:

| Task | Scales | Intervention |
|---|---|---|
| s01 | micro, small | Hard: `PF_M1_T1_Force = 17000` |
| s02 | micro, small | Soft: `PF_M1_T1_Fmax ~ Normal(18500, 3000)` |
| s03 | medium, large | Soft: `PF_M1_T1_Fmax` and `PF_M1_T2_Fmax` |
| s04 | medium, large | Soft on two observed variables + hidden constant on `MV1_Emv` |

For each dataset the notebook writes `observed_data.csv` (observable nodes only),
`intervention_mask.csv`, and `interventions.json`.

**Edit the configuration cell below, then Run All.**

In [1]:
# ── The only cell you need to edit ────────────────────────────────────────────

# Choose which scales to generate. Tasks are automatically filtered to their
# compatible scales (s01/s02 → micro/small; s03/s04 → medium/large).
SCALES      = ["micro"]   # any subset of ["micro", "small", "medium", "large"]
SEED        = 42
N_SAMPLES   = 10_000
OUTPUT_ROOT = "output/causalman_rca"

# ─────────────────────────────────────────────────────────────────────────────

In [2]:
import json
import os

from sympy.stats import Normal

from causalman import CausalMan

# ── Task registry ─────────────────────────────────────────────────────────────
# Each entry defines one RCA scenario. The "scales" list controls which
# CausalMan variants are used for that task.
#
# kind="constant" → hard/atomic intervention (do(X = value))
# kind="normal"   → soft/stochastic intervention (do(X ~ Normal(mean, std)))
TASKS = [
    {
        "task_id": "s01",
        "slug":    "force_17000",
        "scales":  ["micro", "small"],
        "interventions": [
            {"variable": "PF_M1_T1_Force", "kind": "constant", "value": 17000.0},
        ],
    },
    {
        "task_id": "s02",
        "slug":    "fmax_normal",
        "scales":  ["micro", "small"],
        "interventions": [
            {"variable": "PF_M1_T1_Fmax", "kind": "normal", "mean": 18500.0, "std": 3000.0},
        ],
    },
    {
        "task_id": "s03",
        "slug":    "two_fmax_normal",
        "scales":  ["medium", "large"],
        "interventions": [
            {"variable": "PF_M1_T1_Fmax", "kind": "normal", "mean": 18500.0, "std": 3000.0},
            {"variable": "PF_M1_T2_Fmax", "kind": "normal", "mean": 19500.0, "std": 4000.0},
        ],
    },
    {
        "task_id": "s04",
        "slug":    "mixed_with_hidden_emv",
        "scales":  ["medium", "large"],
        "interventions": [
            # Two observed soft interventions
            {"variable": "MV2_DmvMax",     "kind": "normal",   "mean": 4.7,     "std": 1.0},
            {"variable": "PF_M1_T1_Force", "kind": "normal",   "mean": 16500.0, "std": 3000.0},
            # One hidden hard intervention (MV1_Emv is a latent node)
            {"variable": "MV1_Emv",        "kind": "constant", "value": 190000.0},
        ],
    },
]

In [3]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ── Step 1: establish observable columns from a clean observational run ────────
# Run once per scale to pin a consistent observable column list.
# Two filters are applied (matching the observational notebook):
#   (a) the DAG must mark the node as observable — attribute is Boolean True
#       or the string "Observable" depending on the code path;
#   (b) the column must not be constant in the observational data — constant
#       columns carry no information and are excluded.
active_scales = sorted({s for task in TASKS for s in task["scales"] if s in SCALES})
observable_by_scale = {}
for scale in active_scales:
    sim = CausalMan(
        name=f"causalman_{scale}", seed=SEED, batch_multiplier=1,
        parallelize=True, save_path=os.path.join(OUTPUT_ROOT, f"_obs_{scale}"),
    )
    df_obs, _, _, dag = sim.sample()

    dag_observable = [
        n for n, d in dag.nodes(data=True)
        if d.get("Observable") in (True, "Observable") and n in df_obs.columns
    ]
    observable = [col for col in dag_observable if df_obs[col].nunique(dropna=False) > 1]
    observable_by_scale[scale] = observable

    print(f"{scale}: {len(observable)} observable columns  "
          f"({len(dag_observable) - len(observable)} constant columns removed)")

# ── Step 2: generate each interventional dataset ───────────────────────────────
for task in TASKS:
    for scale in [s for s in task["scales"] if s in SCALES]:
        observable = observable_by_scale[scale]
        dataset_id = f"rca_{task['task_id']}_{scale}_{task['slug']}"
        out_dir = os.path.join(OUTPUT_ROOT, dataset_id)
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n── {dataset_id} ──")

        # Build the intervention dict that the simulator accepts.
        # Constants are plain floats; soft interventions use a sympy Normal distribution.
        intervention_dict = {}
        for iv in task["interventions"]:
            if iv["kind"] == "constant":
                intervention_dict[iv["variable"]] = iv["value"]
            else:
                intervention_dict[iv["variable"]] = Normal(iv["variable"], iv["mean"], iv["std"])

        sim = CausalMan(
            name=f"causalman_{scale}", seed=SEED, batch_multiplier=1,
            parallelize=True, save_path=os.path.join(out_dir, "_simulator"),
        )
        sim.intervention_dict = intervention_dict
        df, intervention_mask, _, _ = sim.sample()

        if len(df) < N_SAMPLES:
            raise RuntimeError(
                f"Simulator produced {len(df)} rows but {N_SAMPLES} are required. "
                "Increase batch_multiplier or reduce N_SAMPLES."
            )

        # Sample the same row indices for both files so they stay aligned.
        idx = df.sample(n=N_SAMPLES, random_state=SEED).index
        df.loc[idx, observable].to_csv(os.path.join(out_dir, "observed_data.csv"), index=False)
        intervention_mask.loc[idx].to_csv(os.path.join(out_dir, "intervention_mask.csv"), index=False)

        with open(os.path.join(out_dir, "interventions.json"), "w") as f:
            json.dump(task["interventions"], f, indent=2)

        targets = [iv["variable"] for iv in task["interventions"]]
        print(f"  {N_SAMPLES:,} rows  |  {len(observable)} observable columns")
        print(f"  intervention targets: {targets}")

print(f"\nDone → {os.path.abspath(OUTPUT_ROOT)}")

Starting simulation for production line 0 out of 1
Finished sampling
micro: 24 observable columns  (29 constant columns removed)

── rca_s01_micro_force_17000 ──
Starting simulation for production line 0 out of 1
Finished sampling
  10,000 rows  |  24 observable columns
  intervention targets: ['PF_M1_T1_Force']

── rca_s02_micro_fmax_normal ──
Starting simulation for production line 0 out of 1
Finished sampling
  10,000 rows  |  24 observable columns
  intervention targets: ['PF_M1_T1_Fmax']

Done → c:\Users\tan2rng\CausalMan\src\output\causalman_rca
